In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Changed model type from AutoModelForSequenceClassification

model_name = "Narrativa/mT5-base-finetuned-tydiQA-xqa"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print(f"\nModel loaded on: {device}")

# --- Generative Question Answering Example ---
print("\n--- Generative Question Answering Example ---")

def get_generative_answer(question, context, model, tokenizer, device, max_length=50):
    # The format is "question: <question> context: <context>"
    input_text = f"question: {question} context: {context}"

    # Tokenize the input
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=True,
        padding='max_length', # Or 'longest'
        max_length=512 # mT5 base max sequence length
    ).to(device)

    output_tokens = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_length=max_length,
        num_beams=4, # Beam search for better quality
        early_stopping=True,
        no_repeat_ngram_size=2 # To avoid repetitive text
    )

    # Decode the generated tokens
    answer = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    return answer

# Example usage
question1 = "Kto je autorom tohto textu?"
context1 = "Tento text napísal Ján Novák ako esej o slovenskej literatúre."
answer1 = get_generative_answer(question1, context1, model, tokenizer, device)
print(f"\nQuestion: '{question1}'")
print(f"Context: '{context1}'")
print(f"Answer: '{answer1}'")

question2 = "Kedy sa konala akcia?"
context2 = "Konferencia sa uskutočnila 15. mája 2023 v Bratislave."
answer2 = get_generative_answer(question2, context2, model, tokenizer, device)
print(f"\nQuestion: '{question2}'")
print(f"Context: '{context2}'")
print(f"Answer: '{answer2}'")

question3 = "Aký je hlavný cieľ projektu?"
context3 = "Projekt má za cieľ zlepšiť prístup k vzdelaniu pre znevýhodnené skupiny prostredníctvom online kurzov a mentoringu. Zameriava sa na rozvoj digitálnych zručností a podporu zamestnateľnosti." # The project aims to improve access to education for disadvantaged groups through online courses and mentoring. It focuses on the development of digital skills and support for employability.
answer3 = get_generative_answer(question3, context3, model, tokenizer, device)
print(f"\nQuestion: '{question3}'")
print(f"Context: '{context3}'")
print(f"Answer: '{answer3}'")

Loading Narrativa/mT5-base-finetuned-tydiQA-xqa...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/408 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]


Model loaded on: cuda

--- Generative Question Answering Example ---

Question: 'Kto je autorom tohto textu?'
Context: 'Tento text napísal Ján Novák ako esej o slovenskej literatúre.'
Answer: 'Ján Novák'

Question: 'Kedy sa konala akcia?'
Context: 'Konferencia sa uskutočnila 15. mája 2023 v Bratislave.'
Answer: '15. mája 2023'

Question: 'Aký je hlavný cieľ projektu?'
Context: 'Projekt má za cieľ zlepšiť prístup k vzdelaniu pre znevýhodnené skupiny prostredníctvom online kurzov a mentoringu. Zameriava sa na rozvoj digitálnych zručností a podporu zamestnateľnosti.'
Answer: 'zlepšiť prístup k vzdelaniu pre znevýhodnené skupiny prostredníctvom online kurzov a mentoringu'
